# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [11]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [10]:
import pandas as pd

import pandas as pd

# Load the uploaded Spaceship Titanic dataset
df = pd.read_csv("C:/Users/Beheerder/Desktop/IronHack/week17/lab-ensemble/spaceship_titanic.csv")
df.head()



,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [23]:
# Drop irrelevant or high-missing columns if needed (e.g., Name, Cabin)
df = df.drop(columns=['Name', 'Cabin', 'PassengerId'])

# Fill missing values
df = df.fillna(df.median(numeric_only=True))
df = df.fillna('Unknown')  # For categorical variables

# Convert categorical to dummy/one-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True)


In [27]:
# Split Features and Target
# Ensure target is numerical (0/1)
df_encoded['Transported'] = df_encoded['Transported'].astype(int)

X = df_encoded.drop('Transported', axis=1)
y = df_encoded['Transported']


Now perform the same as before:
- Feature Scaling
- Feature Selection


 # Feature Scaling

In [32]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Feature Selection

In [31]:
from sklearn.feature_selection import SelectKBest, f_classif

# Select top k features based on ANOVA F-statistic
selector = SelectKBest(score_func=f_classif, k=10)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# Get the selected feature names
selected_features = X.columns[selector.get_support()]
print("Selected features:", selected_features.tolist())


Selected features: ['Age', 'RoomService', 'FoodCourt', 'Spa', 'VRDeck', 'HomePlanet_Europa', 'HomePlanet_Mars', 'CryoSleep_True', 'Destination_TRAPPIST-1e', 'VIP_True']


In [29]:
print(df.columns)


Index(['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService',
       'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported'],
      dtype='object')


**Perform Train Test Split**

In [33]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop('Transported', axis=1)
y = df['Transported']

# Perform the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (6954, 10)
Test shape: (1739, 10)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [37]:
print(df.columns.tolist())


['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported']


In [46]:
# Re-importing required libraries after code execution environment reset
import pandas as pd
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Reload the dataset
df = pd.read_csv('/mnt/data/spaceship_titanic.csv')

# Clean and prepare the data
df_cleaned = df.drop(['PassengerId', 'Cabin', 'Name'], axis=1)
df_cleaned.fillna(method='ffill', inplace=True)
df_encoded = pd.get_dummies(df_cleaned, drop_first=True)

X = df_encoded.drop('Transported', axis=1)
y = df_encoded['Transported']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Bagging Classifier
bag_clf = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)
bag_clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = bag_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

accuracy, report


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/spaceship_titanic.csv'

In [43]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True,      # Bagging
    n_jobs=-1,
    random_state=42
)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)


- Random Forests

In [47]:
from sklearn.ensemble import RandomForestClassifier

# Train a Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_clf.fit(X_train, y_train)

# Predict and evaluate
rf_pred = rf_clf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_report = classification_report(y_test, rf_pred)

rf_accuracy, rf_report


(0.7745830937320299,
 '              precision    recall  f1-score   support\n\n       False       0.78      0.75      0.77       861\n        True       0.77      0.79      0.78       878\n\n    accuracy                           0.77      1739\n   macro avg       0.77      0.77      0.77      1739\nweighted avg       0.77      0.77      0.77      1739\n')

- Gradient Boosting

In [48]:
from sklearn.ensemble import GradientBoostingClassifier

# Train Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_clf.fit(X_train, y_train)

# Predict and evaluate
gb_pred = gb_clf.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)
gb_report = classification_report(y_test, gb_pred)

gb_accuracy, gb_report


(0.7797584818861415,
 '              precision    recall  f1-score   support\n\n       False       0.81      0.72      0.77       861\n        True       0.76      0.83      0.79       878\n\n    accuracy                           0.78      1739\n   macro avg       0.78      0.78      0.78      1739\nweighted avg       0.78      0.78      0.78      1739\n')

- Adaptive Boosting

In [49]:
from sklearn.ensemble import AdaBoostClassifier

# Train AdaBoost Classifier
ada_clf = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
ada_clf.fit(X_train, y_train)

# Predict and evaluate
ada_pred = ada_clf.predict(X_test)
ada_accuracy = accuracy_score(y_test, ada_pred)
ada_report = classification_report(y_test, ada_pred)

ada_accuracy, ada_report


C:\Users\Beheerder\anaconda3\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


(0.7630822311673375,
 '              precision    recall  f1-score   support\n\n       False       0.75      0.79      0.77       861\n        True       0.78      0.74      0.76       878\n\n    accuracy                           0.76      1739\n   macro avg       0.76      0.76      0.76      1739\nweighted avg       0.76      0.76      0.76      1739\n')

Which model is the best and why?

### Model Comparison
Model	Accuracy	Strengths	Weaknesses
Gradient Boosting	77.98%	High precision/recall balance, handles complex patterns well	Slower training, sensitive to noisy data
Random Forest	77.46%	Robust to overfitting, good general performance, interpretable	Less optimized than boosting in some cases
AdaBoost	76.31%	Good with simpler base models, can handle bias in data	Slightly lower performance, sensitive to noisy data and outliers
Bagging (Decision Trees)	(not fully run due to env issue)	Reduces variance by averaging multiple models	May underperform if base learners are too weak or biased

🏆 Best Model: Gradient Boosting Classifier
Why it's the best:
✅ Highest accuracy (77.98%)

✅ Balanced precision and recall

✅ Strong performance on both classes (unlike AdaBoost which leaned slightly)

✅ Can model complex non-linear relationships

📌 When to Use Which Model:
Situation	Recommended Model
Best overall accuracy	Gradient Boosting
Speed and simplicity	Random Forest
High bias/low variance problems	AdaBoost
Interpretability and training speed	Random Forest